# XGBoost (Pipeline B)

**Model**: `xgboost.XGBRegressor` / **Target**: gdpc1
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 7
**Scaling**: NO / **HP tuning**: YES / **10-model averaging**: YES
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025
**Grid**: max_depth x eta x n_estimators x reg_lambda (24 combos), TimeSeriesSplit(5)


In [1]:
import sys, os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
                     get_features, load_data, PREDICTIONS_DIR, TARGET)

SEED=42; np.random.seed(SEED)
TRAIN_START='1959-01-01'; TRAIN_END='2007-12-31'; VAL_END='2016-12-31'
TEST_START='2017-01-01'; TEST_END='2025-12-31'
N_LAGS=4; N_MODELS=10
VINTAGES={'m1':-2,'m2':-1,'m3':0,'post1':1}
print('XGBoost -- Cat3, n_lags={}, 10-model avg'.format(N_LAGS))


XGBoost -- Cat3, n_lags=4, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print('Features: {}'.format(len(features)))

# Phase A: HP tuning on validation (2008-2016), TimeSeriesSplit(5)
tune_data = monthly[(monthly['date']>=TRAIN_START)&(monthly['date']<=VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how='any').reset_index(drop=True)
feature_cols = [c for c in tune_flat.columns if c != 'date' and c != TARGET
                and any(c == f or c.startswith(f + '_') for f in features)]

print('Tuning HPs on validation window...')
tscv = TimeSeriesSplit(n_splits=5)
best_score = np.inf
best_params = {'max_depth': 3, 'eta': 0.1, 'n_estimators': 200, 'reg_lambda': 1}
for md in [3, 5]:
    for eta in [0.05, 0.1]:
        for ne in [200, 500]:
            for rl in [0.1, 1, 10]:
                scores = []
                for tr, va in tscv.split(tune_flat):
                    m = XGBRegressor(n_estimators=ne, max_depth=md, eta=eta,
                                     reg_lambda=rl, reg_alpha=0,
                                     random_state=SEED, verbosity=0)
                    m.fit(tune_flat[feature_cols].values[tr], tune_flat[TARGET].values[tr])
                    p = m.predict(tune_flat[feature_cols].values[va])
                    scores.append(np.mean((p - tune_flat[TARGET].values[va]) ** 2))
                if np.mean(scores) < best_score:
                    best_score = np.mean(scores)
                    best_params = {'max_depth': md, 'eta': eta,
                                   'n_estimators': ne, 'reg_lambda': rl}
print('Best XGB params: {}'.format(best_params))


Features: 56


Tuning HPs on validation window...


Best XGB params: {'max_depth': 5, 'eta': 0.1, 'n_estimators': 500, 'reg_lambda': 10}


In [3]:
# Phase B: Rolling test loop with frozen HPs
test_quarters = monthly[(monthly['date']>=TEST_START)&(monthly['date']<=TEST_END)&
                         monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v:[] for v in VINTAGES}; actuals_list=[]; failed=0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date']==pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, fc, N_LAGS
        )
        if len(train_fl)<10: predictions[vn].append(np.nan); continue

        try:
            models=[]
            for k in range(N_MODELS):
                m=XGBRegressor(n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'], eta=best_params['eta'],
                    reg_lambda=best_params['reg_lambda'], reg_alpha=0,
                    random_state=SEED+k, verbosity=0)
                m.fit(train_fl[feature_cols].values, train_fl[TARGET].values)
                models.append(m)
            # Test row with context (P6)
            preds=[m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed+=1
    if (i+1)%6==0: print('  {} / {}'.format(i+1, len(test_quarters)))
print('Done. {} failures.'.format(failed))


  6 / 36


  12 / 36


  18 / 36


  24 / 36


  30 / 36


  36 / 36
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date':test_quarters,'actual':actuals_list,'prediction':predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR,'xgboost_{}.csv'.format(vn)), index=False)
    print('Saved xgboost_{}.csv'.format(vn))


Saved xgboost_m1.csv
Saved xgboost_m2.csv
Saved xgboost_m3.csv
Saved xgboost_post1.csv


In [5]:
def rmse(a,p):
    m=~(np.isnan(a)|np.isnan(p)); return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan
def mae(a,p):
    m=~(np.isnan(a)|np.isnan(p)); return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan
panels={'Pre-COVID':'2017-01-01,2019-12-31','COVID':'2020-04-01,2021-12-31',
        'Post-COVID':'2022-01-01,2025-12-31','Full':'2017-01-01,2025-12-31'}
a=np.array(actuals_list); d=pd.to_datetime(test_quarters)
print('XGB best params: {}'.format(best_params))
for pn,rng in panels.items():
    ps,pe=rng.split(','); m=(d>=ps)&(d<=pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp=np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}'.format(vn,rmse(a[m],pp[m]),mae(a[m],pp[m])))


XGB best params: {'max_depth': 5, 'eta': 0.1, 'n_estimators': 500, 'reg_lambda': 10}
--- Pre-COVID ---
  m1  RMSFE=0.002766  MAE=0.002207
  m2  RMSFE=0.002756  MAE=0.002194
  m3  RMSFE=0.002734  MAE=0.002203
  post1  RMSFE=0.002690  MAE=0.002169
--- COVID ---
  m1  RMSFE=0.042895  MAE=0.026799
  m2  RMSFE=0.042812  MAE=0.026759
  m3  RMSFE=0.042582  MAE=0.026604
  post1  RMSFE=0.042491  MAE=0.026438
--- Post-COVID ---
  m1  RMSFE=0.004563  MAE=0.003444
  m2  RMSFE=0.004595  MAE=0.003466
  m3  RMSFE=0.004427  MAE=0.003306
  post1  RMSFE=0.004386  MAE=0.003312
--- Full ---
  m1  RMSFE=0.019544  MAE=0.008064
  m2  RMSFE=0.019512  MAE=0.008062
  m3  RMSFE=0.019395  MAE=0.007964
  post1  RMSFE=0.019353  MAE=0.007926
